# 课后练习解答（06.04_baseline_benchmark）

本解答对应章节课后练习，共 15 题。

### 问题1（单选题）

**题目：** NPU 计时为什么必须在 timer.stop() 前调用 synchronize？
A. NPU 异步执行，需要等待 kernel 完成
B. 释放显存
C. 触发编译
D. 提高精度

**解答：** A

**解析：** 不同步会记录到未执行完成的时间点，严重低估耗时。


### 问题2（单选题）

**题目：** warmup 主要消除？
A. 首次构图、编译与缓存预热
B. 数据下载
C. 训练 loss
D. 模型保存

**解答：** A

**解析：** 首次迭代包含构图编译与内存分配等一次性开销。


### 问题3（多选题）

**题目：** 基准测试需固定？
A. batch_size
B. 输入 shape/dtype
C. warmup/repeats
D. 同步方式

**解答：** ABCD

**解析：** 任一变量变化都会破坏 baseline 与 optimized 的可比性。


### 问题4（多选题）

**题目：** 峰值显存测量顺序正确的是？
A. reset_peak_memory_stats
B. 执行一次前向
C. synchronize
D. max_memory_allocated

**解答：** ABCD

**解析：** 先清空峰值，再执行、同步、读取，顺序不能颠倒。


### 问题5（判断题）

**题目：** time.perf_counter() 可直接测量 NPU kernel 的真实耗时。

**解答：** 错

**解析：** NPU kernel 异步执行，必须配合 synchronize 才能得到真实耗时。


### 问题6（判断题）

**题目：** 增大 repeats 不会降低真实耗时，但能提高统计稳定性。

**解答：** 对

**解析：** 重复测量只改善估计精度，不改变系统实际性能。


### 问题7（填空题）

**题目：** 读取 NPU 峰值显存的 API 是 ____。

**解答：** torch.npu.max_memory_allocated()


### 问题8（填空题）

**题目：** 吞吐计算：throughput = ____。

**解答：** batch_size * repeats / total_time（images/s）


### 问题9（简答题）

**题目：** 为什么首次迭代通常最慢？

**解答：** 首次执行包含算子构图、内核编译、内存分配和缓存预热，后续迭代复用已编译内核与缓存。


### 问题10（简答题）

**题目：** 如何判断数据加载是性能瓶颈？

**解答：** 用 profiler 观察算子间隙与 DataLoader gap；若 NPU 等待时间长、CPU 预处理热点明显、增加 workers 后吞吐上升，则数据加载是瓶颈。


### 问题11（代码设计题）

**题目：** 编写 benchmark_forward(model, x, warmup, repeats)，返回 ms_per_iter 与 throughput。

**解答：** ```python
def benchmark_forward(model, x, warmup=10, repeats=50):
    model.eval()
    with torch.no_grad():
        for _ in range(warmup): model(x)
        torch.npu.synchronize()
        start = time.perf_counter()
        for _ in range(repeats): model(x)
        torch.npu.synchronize()
        total = time.perf_counter() - start
    ms_per_iter = total / repeats * 1000
    throughput = x.size(0) * repeats / total
    return ms_per_iter, throughput
```


### 问题12（单选题）

**题目：** NPU 利用率低且 CPU 高，最可能？
A. 数据加载瓶颈
B. 算子计算密集
C. 显存不足
D. 模型过大

**解答：** A

**解析：** CPU 忙于取数、NPU 等待，说明供数速度是瓶颈。


### 问题13（多选题）

**题目：** results JSON 应包含？
A. ms_per_iter
B. throughput
C. peak_memory_mb
D. device/版本/输入信息

**解答：** ABCD

**解析：** 只有记录完整环境与输入信息，结果才可复现可对比。


### 问题14（判断题）

**题目：** baseline 与 optimized 必须使用相同输入与 repeats。

**解答：** 对

**解析：** 不同输入或测量次数会引入系统性偏差。


### 问题15（简答题）

**题目：** 如何降低基准测量噪声？请至少给出 4 种方法。

**解答：** 1) 增加 warmup 与 repeats；2) 使用中位数而非单次值；3) 固定 CPU 频率/关闭后台任务；4) 保证每次测量前缓存状态一致；5) 多个 batch 交替测量并报告置信区间。
